# Extended Text Model Comparison

## Data Preparation

In [1]:
import json
import os
from pathlib import Path
import time
from collections.abc import Callable

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import numpy as np
import pandas as pd
import tensorflow as tf
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "transaction_text_dataset.csv"
RESULTS_PATH = PROJECT_ROOT / "data" / "processed" / "text_model_comparison_results.json"
RANDOM_STATE = 42
MAX_TOKENS = 5_000
SEQUENCE_LENGTH = 80
EMBEDDING_DIMENSION = 32
BATCH_SIZE = 128
MAX_EPOCHS = 5
PRETRAINED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
tf.keras.utils.set_random_seed(RANDOM_STATE)

In [2]:
data = pd.read_csv(DATA_PATH)
X = data["transaction_text"].astype(str)
y = data["is_fraud"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

class_counts = y_train.value_counts().sort_index()
class_weights = {
    class_label: len(y_train) / (len(class_counts) * count)
    for class_label, count in class_counts.items()
}
print(f"Training rows: {len(X_train):,}; test rows: {len(X_test):,}")
print("Class weights:", class_weights)

Training rows: 24,000; test rows: 6,000
Class weights: {0: 0.5292405398253506, 1: 9.049773755656108}


In [3]:
vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=MAX_TOKENS,
    output_mode="int",
    output_sequence_length=SEQUENCE_LENGTH,
)
vectorizer.adapt(tf.data.Dataset.from_tensor_slices(X_train).batch(256))
X_train_sequences = vectorizer(np.asarray(X_train)).numpy()
VOCABULARY_SIZE = len(vectorizer.get_vocabulary())
print(f"Vocabulary size: {VOCABULARY_SIZE:,}")

Vocabulary size: 5,000


## Neural Model Definitions

In [4]:
def compile_model(model: tf.keras.Model) -> tf.keras.Model:
    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=[tf.keras.metrics.AUC(name="roc_auc")],
    )
    return model

def embedding_input() -> tuple[tf.keras.layers.Input, tf.Tensor]:
    inputs = tf.keras.Input(shape=(SEQUENCE_LENGTH,), dtype="int64")
    embeddings = tf.keras.layers.Embedding(VOCABULARY_SIZE, EMBEDDING_DIMENSION)(inputs)
    return inputs, embeddings

def build_cnn() -> tf.keras.Model:
    inputs, embeddings = embedding_input()
    features = tf.keras.layers.Conv1D(32, 3, activation="relu")(embeddings)
    features = tf.keras.layers.GlobalMaxPooling1D()(features)
    features = tf.keras.layers.Dropout(0.2)(features)
    outputs = tf.keras.layers.Dense(1, activation="sigmoid")(features)
    return compile_model(tf.keras.Model(inputs, outputs, name="cnn"))

def build_simple_rnn() -> tf.keras.Model:
    inputs, embeddings = embedding_input()
    features = tf.keras.layers.SimpleRNN(24, dropout=0.1)(embeddings)
    outputs = tf.keras.layers.Dense(1, activation="sigmoid")(features)
    return compile_model(tf.keras.Model(inputs, outputs, name="simple_rnn"))

def build_lstm() -> tf.keras.Model:
    inputs, embeddings = embedding_input()
    features = tf.keras.layers.LSTM(24, dropout=0.1)(embeddings)
    outputs = tf.keras.layers.Dense(1, activation="sigmoid")(features)
    return compile_model(tf.keras.Model(inputs, outputs, name="lstm"))

def build_attention_transformer() -> tf.keras.Model:
    inputs, embeddings = embedding_input()
    attention = tf.keras.layers.MultiHeadAttention(num_heads=2, key_dim=16)(embeddings, embeddings)
    features = tf.keras.layers.Add()([embeddings, attention])
    features = tf.keras.layers.LayerNormalization()(features)
    features = tf.keras.layers.GlobalAveragePooling1D()(features)
    features = tf.keras.layers.Dropout(0.2)(features)
    outputs = tf.keras.layers.Dense(1, activation="sigmoid")(features)
    return compile_model(tf.keras.Model(inputs, outputs, name="attention_transformer"))

## CNN, RNN, LSTM, and Attention Comparison

In [5]:
def evaluate_probabilities(
    model_name: str,
    probabilities: np.ndarray,
    inference_seconds: float,
    epochs_trained: int | None = None,
) -> dict[str, float | int | str]:
    predictions = (probabilities >= 0.5).astype(int)
    return {
        "model": model_name,
        "accuracy": float(accuracy_score(y_test, predictions)),
        "precision": float(precision_score(y_test, predictions, zero_division=0)),
        "recall": float(recall_score(y_test, predictions, zero_division=0)),
        "f1": float(f1_score(y_test, predictions, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_test, probabilities)),
        "inference_time_seconds": float(inference_seconds),
        "epochs_trained": epochs_trained,
    }

def train_neural_model(
    model_name: str,
    builder: Callable[[], tf.keras.Model],
) -> dict[str, float | int | str]:
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(RANDOM_STATE)
    model = builder()
    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=1, restore_best_weights=True
    )
    history = model.fit(
        X_train_sequences,
        y_train.to_numpy(),
        validation_split=0.15,
        epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,
        class_weight=class_weights,
        callbacks=[early_stopping],
        verbose=2,
    )
    start_time = time.perf_counter()
    test_sequences = vectorizer(np.asarray(X_test)).numpy()
    probabilities = model.predict(test_sequences, batch_size=BATCH_SIZE, verbose=0).ravel()
    inference_seconds = time.perf_counter() - start_time
    return evaluate_probabilities(
        model_name, probabilities, inference_seconds, len(history.history["loss"])
    )

results = []
for name, builder in [
    ("CNN", build_cnn),
    ("Simple RNN", build_simple_rnn),
    ("LSTM", build_lstm),
    ("Attention/Transformer", build_attention_transformer),
]:
    print(f"Training {name}...")
    results.append(train_neural_model(name, builder))

Training CNN...


Epoch 1/5


160/160 - 4s - 22ms/step - loss: 0.6818 - roc_auc: 0.5801 - val_loss: 0.6542 - val_roc_auc: 0.7270


Epoch 2/5


160/160 - 1s - 9ms/step - loss: 0.6375 - roc_auc: 0.6963 - val_loss: 0.5811 - val_roc_auc: 0.7302


...

160/160 - 4s - 23ms/step - loss: 0.6360 - roc_auc: 0.6852 - val_loss: 0.3532 - val_roc_auc: 0.7033


Epoch 3/5


160/160 - 4s - 25ms/step - loss: 0.5538 - roc_auc: 0.7870 - val_loss: 0.2691 - val_roc_auc: 0.6891


Epoch 4/5


160/160 - 4s - 23ms/step - loss: 0.4814 - roc_auc: 0.8468 - val_loss: 0.2769 - val_roc_auc: 0.6433


## Pretrained MiniLM

In [6]:
pretrained_encoder = SentenceTransformer(PRETRAINED_MODEL_NAME, device="cpu")
train_embeddings = pretrained_encoder.encode(
    X_train.tolist(), batch_size=64, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True
)
pretrained_classifier = LogisticRegression(
    class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE, n_jobs=2
)
pretrained_classifier.fit(train_embeddings, y_train)

start_time = time.perf_counter()
test_embeddings = pretrained_encoder.encode(
    X_test.tolist(), batch_size=64, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True
)
pretrained_probabilities = pretrained_classifier.predict_proba(test_embeddings)[:, 1]
pretrained_inference_seconds = time.perf_counter() - start_time
results.append(evaluate_probabilities(
    "Pretrained MiniLM", pretrained_probabilities, pretrained_inference_seconds
))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\Priya Koma\Desktop\AI_ML_Assessments\ai_fraud_detection_poc\.venv\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Priya Koma\Desktop\AI_ML_Assessments\ai_fraud_detection_poc\.venv\hf_cache\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/375 [00:00<?, ?it/s]

C:\Users\Priya Koma\Desktop\AI_ML_Assessments\ai_fraud_detection_poc\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=2', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Batches:   0%|          | 0/94 [00:00<?, ?it/s]

## Final Comparison Results

In [7]:
comparison = pd.DataFrame(results).sort_values("f1", ascending=False).reset_index(drop=True)
best_f1 = comparison.loc[comparison["f1"].idxmax()]
best_roc_auc = comparison.loc[comparison["roc_auc"].idxmax()]
comparison_artifact = {
    "dataset": DATA_PATH.name,
    "random_state": RANDOM_STATE,
    "models": comparison.where(pd.notna(comparison), None).to_dict(orient="records"),
    "best_by_f1": {"model": best_f1["model"], "f1": float(best_f1["f1"])},
    "best_by_roc_auc": {"model": best_roc_auc["model"], "roc_auc": float(best_roc_auc["roc_auc"])},
}
with RESULTS_PATH.open("w", encoding="utf-8") as results_file:
    json.dump(comparison_artifact, results_file, indent=2)

display(comparison.style.format({
    "accuracy": "{:.4f}",
    "precision": "{:.4f}",
    "recall": "{:.4f}",
    "f1": "{:.4f}",
    "roc_auc": "{:.4f}",
    "inference_time_seconds": "{:.4f}",
}))
print(f"Saved comparison results to: {RESULTS_PATH}")
print(f"Best by F1: {best_f1['model']} ({best_f1['f1']:.4f})")
print(f"Best by ROC-AUC: {best_roc_auc['model']} ({best_roc_auc['roc_auc']:.4f})")

,model,accuracy,precision,recall,f1,roc_auc,inference_time_seconds,epochs_trained
0,Pretrained MiniLM,0.6192,0.0875,0.6235,0.1534,0.6695,143.4486,nan
1,CNN,0.7673,0.0920,0.3614,0.1467,0.6176,0.2909,5.000000
2,Attention/Transformer,0.9103,0.1424,0.1235,0.1323,0.6803,0.7122,4.000000
3,Simple RNN,0.8678,0.1033,0.1807,0.1314,0.6162,1.3358,5.000000
4,LSTM,0.7593,0.0807,0.3223,0.1291,0.5809,1.1517,4.000000


Saved comparison results to: C:\Users\Priya Koma\Desktop\AI_ML_Assessments\ai_fraud_detection_poc\data\processed\text_model_comparison_results.json
Best by F1: Pretrained MiniLM (0.1534)
Best by ROC-AUC: Attention/Transformer (0.6803)
